# 📦 Lab 3: Dynamic Payload Drops, CG Shifts & Parallel-Axis Inertia
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blaze505050/drone-digital-twin/blob/main/examples/labs/Lab3_Dynamic_Payload_Release_and_Inertia.ipynb)

Welcome to **Lab 3 of the DronePy Aerospace & Robotics Curriculum**!
In this lab, you will study rigid-body kinematics under sudden discrete mass changes, calculate center-of-gravity (CG) shifts, apply the **Parallel-Axis (Huygens-Steiner) Theorem** to update the 3x3 inertia tensor, and observe closed-loop transient recovery during an in-flight cargo release.

---
### 🎯 Learning Objectives
1. Formulate the mass matrix and 3x3 inertia tensor for a multirotor with external payloads.
2. Calculate the instantaneous center of gravity shift and parallel-axis inertia transformation.
3. Schedule discrete in-flight payload drop events using DronePy's `Scenario` engine.
4. Quantify vertical acceleration impulse, altitude overshoot, and attitude stabilization settling time.


In [ ]:
# Setup dependencies
try:
    import dronepy
except ImportError:
    !pip install -q git+https://github.com/blaze505050/drone-digital-twin.git
    import dronepy

import numpy as np
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

print(f"DronePy Version: {dronepy.__version__}")


---
## 1. Mathematical Theory: Parallel-Axis Theorem & Center of Gravity

### A. Center of Gravity Shift
When a payload of mass $m_p$ at position vector $\mathbf{r}_p = [x_p, y_p, z_p]^T$ is attached to a vehicle of base mass $m_0$:
$$M_{\text{total}} = m_0 + m_p$$
$$\mathbf{r}_{\text{cg}} = \frac{m_0 \mathbf{r}_0 + m_p \mathbf{r}_p}{M_{\text{total}}}$$

### B. Parallel-Axis Theorem in Tensor Form
The 3x3 moment of inertia matrix transforms as:
$$\mathbf{J}_{\text{new}} = \mathbf{J}_{\text{base}} + m_p \left( \|\mathbf{r}_p\|^2 \mathbf{I}_{3\times 3} - \mathbf{r}_p \mathbf{r}_p^T \right)$$

In component form:
$$I_{xx}' = I_{xx} + m_p (y_p^2 + z_p^2)$$
$$I_{yy}' = I_{yy} + m_p (x_p^2 + z_p^2)$$
$$I_{zz}' = I_{zz} + m_p (x_p^2 + y_p^2)$$
$$I_{xy}' = I_{xy} - m_p x_p y_p$$

### C. Sudden Mass Release Transient Dynamics
When the payload is released at $t = t_{\text{drop}}$, the upward thrust immediately exceeds the reduced weight:
$$a_z(t_{\text{drop}}^+) = \frac{T_{\text{hover}} - m_0 g}{m_0} = \frac{(m_0 + m_p) g - m_0 g}{m_0} = g \cdot \frac{m_p}{m_0}$$
This produces an instantaneous positive vertical climb acceleration that the altitude controller must reject.


In [ ]:
# 2. Inspecting Inertia Shift with DronePy
base_drone = dronepy.Drone.quadcopter(mass=1.50)
print(f"Base Vehicle Mass: {base_drone.mass:.2f} kg")
print(f"Base Inertia Diagonal (Ixx, Iyy, Izz): {np.diag(base_drone.inertia_tensor)}")

# Attach a 0.50 kg delivery payload 10 cm below the airframe
base_drone.add_payload(mass_kg=0.50, offset_m=np.array([0.0, 0.0, 0.10]))
print(f"With Payload Mass: {base_drone.mass:.2f} kg")
print(f"Updated Inertia Diagonal:              {np.diag(base_drone.inertia_tensor)}")
print(f"Updated Center of Gravity (m):         {base_drone.center_of_gravity}")


---
## 3. Simulating In-Flight Payload Release
We will now use DronePy's `Scenario` class to schedule a cargo release at $t = 2.0\text{ s}$ during a 5.0 s hover mission.


In [ ]:
# Setup drone with payload
drone = dronepy.Drone.quadcopter(mass=1.50)
drone.add_payload(mass_kg=0.50, offset_m=np.array([0.0, 0.0, 0.10]))

# Create flight scenario with discrete event
scenario = dronepy.Scenario()
scenario.at(2.0).payload_release()

flight = dronepy.Flight(drone=drone, events=scenario, duration=5.0)
res = flight.result

idx_drop = np.searchsorted(res.time, 2.0)
alt_before_drop = -res.pos_ned[idx_drop, 2]
alt_peak = -np.min(res.pos_ned[idx_drop:, 2])
alt_overshoot = alt_peak - alt_before_drop

print(f"Altitude at Release:   {alt_before_drop:.3f} m")
print(f"Peak Altitude:         {alt_peak:.3f} m")
print(f"Release Altitude Bump: {alt_overshoot:.3f} m")
print(f"Final Payload Mass:    {drone.payload_mass:.3f} kg")


---
## 📝 Student Exercise: Asymmetric Cargo Release & Attitude Recovery

### Scenario:
A medical courier drone carries a **$0.40\text{ kg}$** emergency vaccine canister mounted off-center at:
$$\mathbf{r}_{\text{offset}} = [0.06\text{ m (forward)}, \; 0.04\text{ m (right)}, \; 0.08\text{ m (down)}]^T$$
At $t = 2.5\text{ s}$, the package is released. Because the payload was off-center, the sudden release will induce both a vertical climb bump and an attitude torque disturbance.

### Your Tasks:
1. Attach the asymmetric payload to a $1.60\text{ kg}$ quadcopter.
2. Schedule a payload release at $t = 2.5\text{ s}$.
3. Simulate for 6.0 seconds.
4. Calculate:
   - Expected initial vertical acceleration $a_z = g \cdot (m_p / m_0)$
   - Peak roll and pitch attitude error angles during recovery
   - Verify that final payload mass is exactly 0.0 kg


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STUDENT SOLUTION CELL - Simulate asymmetric drop:
# ══════════════════════════════════════════════════════════════════
drone_courier = dronepy.Drone.quadcopter(mass=1.60)
payload_mass = 0.40
r_off = np.array([0.06, 0.04, 0.08])
drone_courier.add_payload(mass_kg=payload_mass, offset_m=r_off)

scenario_med = dronepy.Scenario()
scenario_med.at(2.5).payload_release()

flight_med = dronepy.Flight(drone=drone_courier, events=scenario_med, duration=6.0)
res_med = flight_med.result

expected_az = 9.80665 * (payload_mass / 1.60)
post_drop_mask = res_med.time >= 2.5
max_roll_deg = np.max(np.abs(res_med.euler_deg[post_drop_mask, 0]))
max_pitch_deg = np.max(np.abs(res_med.euler_deg[post_drop_mask, 1]))

print(f"Theoretical Initial Vert Accel: {expected_az:.2f} m/s^2")
print(f"Peak Roll Disturbance:         {max_roll_deg:.2f} deg")
print(f"Peak Pitch Disturbance:        {max_pitch_deg:.2f} deg")

# Verification Assertions
assert expected_az > 2.0, "Initial acceleration should exceed 2 m/s^2"
assert max_pitch_deg > 0.0, "Asymmetric pitch offset must create measurable pitch transient"
assert drone_courier.payload_mass == 0.0, "Payload mass must be zero after release"
print("SUCCESS: Lab 3 payload dynamics verified successfully!")
